# End-to-end Ball-DP demo: MNIST embeddings + ridge prototypes

This notebook is intentionally **not** the official paper experiment runner.  
It is a compact operational demo that does one complete run:

1. Load MNIST embeddings.
2. Choose a Ball radius \(r\) from within-label embedding distances.
3. Train a ridge-prototype release under Ball-DP.
4. Train a standard comparator release.
5. Report utility, Gaussian noise, and theorem-backed ReRo bounds.
6. Build one finite candidate support \(S\subseteq \mathcal B(u,r)\).
7. Run the exact finite-prior MAP reconstruction attack.
8. Report empirical exact-identification and support-level diagnostics.

The official scripts `run_erm_experiment.py`, `run_attack_experiment.py`, `aggregate_erm.py`, and `aggregate_attack.py` do many ablations and aggregate across datasets. This notebook avoids that complexity.

## Threat model and mechanism

Records are embeddings with labels:
\[
z=(x,y),\qquad x\in\mathbb R^d.
\]

The label-preserving metric is
\[
d((x,y),(x',y'))=
\begin{cases}
\|x-x'\|_2,& y=y',\\
\infty,& y\ne y'.
\end{cases}
\]

The local Ball-DP adjacency relation protects single-record replacements with
\[
d(z,z')\le r.
\]

The ridge-prototype model learns one prototype per class:
\[
F_D(\mu)
=
\frac{1}{n}\sum_{i=1}^n \|x_i-\mu_{y_i}\|_2^2
+
\frac{\lambda}{2}\sum_c \|\mu_c\|_2^2.
\]

For class \(c\), the exact optimizer is
\[
\hat\mu_c(D)=\frac{2S_c(D)}{2n_c(D)+\lambda n},
\qquad
S_c(D)=\sum_{i:y_i=c}x_i.
\]

The Gaussian output release is
\[
\widetilde \mu=\hat\mu(D)+\xi,
\qquad
\xi\sim \mathcal N(0,\sigma^2 I).
\]

In [ ]:
# Notebook configuration

from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# If you run this notebook from inside the repo, this is usually unnecessary.
# It makes the notebook robust when launched from a subdirectory.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "quantbayes").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantbayes.ball_dp import (
    attack_convex_ball_output_finite_prior,
    ball_rero,
    diagnose_convex_ball_output_finite_prior,
    fit_convex,
    make_finite_identification_prior,
    select_ball_radius,
    summarize_embedding_ball_radii,
)
from quantbayes.ball_dp.experiments.load_mnist_embeddings import (
    load_or_create_mnist_resnet18_embeddings,
)

# Main privacy/training config.
EPSILON = 8.0
DELTA = 1e-6
LAMBDA = 1e-2
RADIUS_QUANTILE = 0.80
M = 8
RELEASE_SEED = 0

# This demo uses the count-aware ridge sensitivity because label-preserving
# adjacency keeps class counts fixed. Set to "global" to reproduce the
# count-worst-case calibration.
RIDGE_SENSITIVITY_MODE = "count_aware"  # "global" or "count_aware"

# Candidate support construction for the attack demo.
SUPPORT_SELECTION = "farthest"  # "random", "farthest", or "nearest"
ANCHOR_SELECTION = "rare_class" # "random", "rare_class", or "large_bank"

# Keep the demo lighter than the paper experiments. Set these to None for full MNIST.
TRAIN_PER_CLASS = 1000
TEST_PER_CLASS = 500

# Embedding loader config.
DATA_ROOT = "./data"
EMBEDDING_CACHE_PATH = None
ALLOW_CPU_FALLBACK = True
FORCE_RECOMPUTE_EMBEDDINGS = False
EMBEDDING_BATCH_SIZE = 128
NUM_WORKERS = 2

np.random.seed(0)
plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})

## Load MNIST embeddings

The loader caches ResNet18 embeddings. The first run can take time if the cache is absent; later runs should be much faster.

In [ ]:
X_train_j, y_train_j, X_test_j, y_test_j = load_or_create_mnist_resnet18_embeddings(
    data_root=DATA_ROOT,
    batch_size=EMBEDDING_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    require_jax_gpu=not ALLOW_CPU_FALLBACK,
    cache_path=EMBEDDING_CACHE_PATH,
    force_recompute=FORCE_RECOMPUTE_EMBEDDINGS,
)

X_train = np.asarray(jax.device_get(X_train_j), dtype=np.float32)
y_train = np.asarray(jax.device_get(y_train_j), dtype=np.int32).reshape(-1)
X_test = np.asarray(jax.device_get(X_test_j), dtype=np.float32)
y_test = np.asarray(jax.device_get(y_test_j), dtype=np.int32).reshape(-1)

def stratified_take(X, y, per_class, seed=0):
    if per_class is None:
        return X, y
    rng = np.random.default_rng(seed)
    keep = []
    for c in sorted(np.unique(y).tolist()):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        keep.extend(idx[:min(int(per_class), len(idx))].tolist())
    keep = np.asarray(keep, dtype=np.int64)
    rng.shuffle(keep)
    return X[keep], y[keep]

X_train, y_train = stratified_take(X_train, y_train, TRAIN_PER_CLASS, seed=0)
X_test, y_test = stratified_take(X_test, y_test, TEST_PER_CLASS, seed=1)

num_classes = int(len(np.unique(y_train)))
feature_dim = int(X_train.shape[1])
empirical_embedding_bound = float(np.max(np.linalg.norm(X_train, axis=1)))
EMBEDDING_BOUND = max(1.0, empirical_embedding_bound + 1e-6)

print({
    "train_shape": X_train.shape,
    "test_shape": X_test.shape,
    "num_classes": num_classes,
    "feature_dim": feature_dim,
    "empirical_embedding_bound": empirical_embedding_bound,
    "public_embedding_bound_used": EMBEDDING_BOUND,
})

## Choose Ball and standard radii

The Ball radius is the worst classwise \(q80\) within-label distance:
\[
r_{\mathrm{Ball}}=\max_c \widehat Q_{0.80}\{\|x_i-x_j\|_2:y_i=y_j=c\}.
\]

The standard comparator radius is the same-label empirical diameter:
\[
r_{\mathrm{Std}}=\max_c \max_{i,j:y_i=y_j=c}\|x_i-x_j\|_2.
\]

This matches the local-vs-global comparison in the paper. A stricter public-bound comparator would instead use \(2B\).

In [ ]:
radius_report = summarize_embedding_ball_radii(
    X_train,
    y_train,
    quantiles=(0.50, 0.80, 0.95),
    max_exact_pairs=600_000,
    max_sampled_pairs=100_000,
    seed=0,
)

r_ball = select_ball_radius(
    radius_report,
    strategy="max_labelwise_quantile",
    quantile=RADIUS_QUANTILE,
)
r_std = select_ball_radius(
    radius_report,
    strategy="global_max",
    quantile=1.0,
    allow_observed_max=True,
)

radius_df = pd.DataFrame([
    {"quantity": "Ball radius q80", "value": r_ball},
    {"quantity": "Standard same-label empirical diameter", "value": r_std},
    {"quantity": "Public bound comparator 2B", "value": 2.0 * EMBEDDING_BOUND},
    {"quantity": "Std/Ball radius ratio", "value": r_std / r_ball},
])
display(radius_df)

## Train one Ball release and one standard release

Both releases use the same model, same \(\varepsilon,\delta,\lambda\), and same random seed.  
Only the release radius changes.

In [ ]:
def fit_ridge_release(radius, *, name):
    release = fit_convex(
        X_train,
        y_train,
        X_eval=X_test,
        y_eval=y_test,
        model_family="ridge_prototype",
        privacy="ball_dp",
        radius=float(radius),
        standard_radius=float(r_std),
        ridge_sensitivity_mode=RIDGE_SENSITIVITY_MODE,
        lam=LAMBDA,
        epsilon=EPSILON,
        delta=DELTA,
        embedding_bound=EMBEDDING_BOUND,
        num_classes=num_classes,
        max_iter=100,
        solver="lbfgs_fullbatch",
        seed=RELEASE_SEED,
    )
    release.extra["demo_name"] = name
    return release

ball_release = fit_ridge_release(r_ball, name="Ball-DP")
std_release = fit_ridge_release(r_std, name="Standard DP")

def release_row(name, release):
    return {
        "mechanism": name,
        "accuracy": float(release.utility_metrics.get("accuracy", np.nan)),
        "sigma": float(release.privacy.ball.sigma),
        "sensitivity_delta": float(release.sensitivity.delta_ball),
        "radius": float(release.privacy.ball.radius),
        "epsilon_certificate": float(release.privacy.ball.dp_certificates[0].epsilon),
        "delta_certificate": float(release.privacy.ball.dp_certificates[0].delta),
    }

erm_df = pd.DataFrame([
    release_row("Ball-DP", ball_release),
    release_row("Standard DP", std_release),
])
erm_df["sigma_ratio_vs_ball"] = erm_df["sigma"] / float(erm_df.loc[erm_df["mechanism"] == "Ball-DP", "sigma"].iloc[0])
display(erm_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.4), constrained_layout=True)

axes[0].bar(erm_df["mechanism"], erm_df["accuracy"])
axes[0].set_title("Accuracy")
axes[0].set_ylim(0, 1)

axes[1].bar(erm_df["mechanism"], erm_df["sigma"])
axes[1].set_title("Gaussian noise scale")
axes[1].set_ylabel(r"$\sigma$")

axes[2].bar(erm_df["mechanism"], erm_df["sensitivity_delta"])
axes[2].set_title("Sensitivity used for calibration")
axes[2].set_ylabel(r"$\Delta_2$")

plt.show()

## Theorem-backed finite-prior exact-ID bounds

For a uniform finite prior on \(m\) candidates and exact-identification loss, the oblivious baseline is
\[
\kappa = 1/m.
\]

For a Gaussian release with sensitivity \(\Delta_2\), the direct Gaussian bound is
\[
\gamma_{\mathrm{direct}}
=
\Phi\left(\Phi^{-1}(1/m)+\frac{\Delta_2}{\sigma}\right).
\]

The RDP bound is also computed, but for Gaussian output perturbation the direct bound is usually sharper.

In [ ]:
finite_prior = make_finite_identification_prior(
    np.zeros((M, feature_dim), dtype=np.float32),
    weights=None,
)

def bound_row(name, release):
    dp = ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="dp")
    direct = ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="gaussian_direct")
    rdp = ball_rero(release, prior=finite_prior, eta_grid=(0.5,), mode="rdp")
    return {
        "mechanism": name,
        "oblivious_baseline_1_over_m": float(direct.points[0].kappa),
        "generic_dp_bound": float(dp.points[0].gamma_ball),
        "direct_gaussian_bound": float(direct.points[0].gamma_ball),
        "rdp_bound": float(rdp.points[0].gamma_ball),
        "same_noise_standard_bound_if_available": (
            None if direct.points[0].gamma_standard is None
            else float(direct.points[0].gamma_standard)
        ),
    }

bound_df = pd.DataFrame([
    bound_row("Ball-DP", ball_release),
    bound_row("Standard DP", std_release),
])
display(bound_df)

In [ ]:
plot_df = bound_df.set_index("mechanism")[[
    "oblivious_baseline_1_over_m",
    "direct_gaussian_bound",
    "rdp_bound",
]]
plot_df.plot(kind="bar")
plt.ylabel("Exact-ID success upper bound")
plt.ylim(0, 1)
plt.title(f"Theorem-backed bounds, m={M}, epsilon={EPSILON:g}")
plt.xticks(rotation=0)
plt.show()

# Finite-prior MAP attack

Now we build one finite candidate support
\[
S=\{z_1,\dots,z_m\}\subseteq \mathcal B(u,r)
\]
around an anchor/center \(u\).

The anchor \(u\) is used to define the Ball and the known dataset \(D^-\).  
The hidden target \(Z\) is one of the public/test candidates in \(S\), not the anchor itself.

For each candidate \(z_i\), the adversary can compute the deterministic model mean:
\[
\mu_i=\hat\theta(D^-\cup\{z_i\}).
\]

The release distribution under candidate \(z_i\) is
\[
\widetilde\theta\mid Z=z_i
\sim
\mathcal N(\mu_i,\sigma^2 I).
\]

With a uniform prior, the Bayes-optimal exact-identification attack is
\[
\hat z
=
\arg\min_{z_i\in S}
\left\|
\widetilde\theta-\hat\theta(D^-\cup\{z_i\})
\right\|_2^2.
\]

For ridge prototypes with known label \(y\), this is equivalent to nearest-neighbor decoding after the noisy inverse
\[
W_y
=
\frac{2(n_y^-+1)+\lambda n}{2}\widetilde\mu_y-S_y^-
=
Z+\eta_y,
\qquad
\eta_y\sim\mathcal N(0,\tau_y^2I).
\]

In [ ]:
def remove_train_index(X, y, index):
    index = int(index)
    return (
        np.concatenate([X[:index], X[index + 1:]], axis=0),
        np.concatenate([y[:index], y[index + 1:]], axis=0),
    )

def greedy_farthest_positions(vectors, m, rng, anchor_distances=None):
    X = np.asarray(vectors, dtype=np.float32)
    n = len(X)
    if n < m:
        raise ValueError(f"Need at least {m} candidates, got {n}.")
    if anchor_distances is None:
        first = int(rng.integers(0, n))
    else:
        top_k = min(n, max(m, 4 * m))
        tail = np.argsort(-np.asarray(anchor_distances))[:top_k]
        first = int(rng.choice(tail))
    selected = [first]
    min_d = np.linalg.norm(X - X[first][None, :], axis=1)
    min_d[first] = -np.inf
    while len(selected) < m:
        best = float(np.max(min_d))
        ties = np.flatnonzero(np.isclose(min_d, best, rtol=1e-7, atol=1e-7))
        nxt = int(rng.choice(ties)) if len(ties) else int(np.argmax(min_d))
        selected.append(nxt)
        d_new = np.linalg.norm(X - X[nxt][None, :], axis=1)
        min_d = np.minimum(min_d, d_new)
        min_d[selected] = -np.inf
    return np.asarray(selected, dtype=np.int64)

def choose_anchor_and_support(
    X_train,
    y_train,
    X_public,
    y_public,
    *,
    radius,
    m,
    anchor_selection="rare_class",
    support_selection="farthest",
    seed=0,
):
    rng = np.random.default_rng(seed)
    counts = np.bincount(y_train.astype(np.int64), minlength=int(y_train.max()) + 1)

    if anchor_selection == "rare_class":
        order = sorted(
            range(len(X_train)),
            key=lambda i: (int(counts[int(y_train[i])]), float(rng.random())),
        )
    elif anchor_selection == "large_bank":
        order = rng.permutation(len(X_train)).tolist()
    elif anchor_selection == "random":
        order = rng.permutation(len(X_train)).tolist()
    else:
        raise ValueError(anchor_selection)

    best = None
    for anchor_idx in order:
        anchor_x = X_train[anchor_idx]
        anchor_y = int(y_train[anchor_idx])
        same = np.where(y_public == anchor_y)[0]
        if len(same) == 0:
            continue
        dists = np.linalg.norm(X_public[same] - anchor_x[None, :], axis=1)
        keep = dists <= float(radius) + 1e-7
        cand_idx = same[keep]
        cand_dists = dists[keep]
        if len(cand_idx) < m:
            continue

        X_bank = X_public[cand_idx]
        if support_selection == "farthest":
            pos = greedy_farthest_positions(X_bank, m=m, rng=rng, anchor_distances=cand_dists)
        elif support_selection == "nearest":
            pos = np.argsort(cand_dists)[:m]
        elif support_selection == "random":
            pos = rng.permutation(len(X_bank))[:m]
        else:
            raise ValueError(support_selection)

        X_support = X_bank[pos].astype(np.float32)
        y_support = np.full((m,), anchor_y, dtype=np.int32)
        source_ids = [f"test:{int(i)}" for i in cand_idx[pos].tolist()]
        support_dists = cand_dists[pos].astype(np.float32)

        candidate = {
            "anchor_index": int(anchor_idx),
            "anchor_label": anchor_y,
            "anchor_vector": anchor_x.astype(np.float32),
            "X_support": X_support,
            "y_support": y_support,
            "source_ids": source_ids,
            "support_distances_to_anchor": support_dists,
            "bank_size": int(len(cand_idx)),
        }
        if anchor_selection != "large_bank":
            return candidate
        if best is None or candidate["bank_size"] > best["bank_size"]:
            best = candidate

    if best is None:
        raise RuntimeError("Could not find an anchor with enough public candidates inside the Ball.")
    return best

support = choose_anchor_and_support(
    X_train,
    y_train,
    X_test,
    y_test,
    radius=r_ball,
    m=M,
    anchor_selection=ANCHOR_SELECTION,
    support_selection=SUPPORT_SELECTION,
    seed=0,
)

support_info = {
    "anchor_index": support["anchor_index"],
    "anchor_label": support["anchor_label"],
    "support_size": len(support["X_support"]),
    "candidate_bank_size": support["bank_size"],
    "support_min_distance_to_anchor": float(np.min(support["support_distances_to_anchor"])),
    "support_mean_distance_to_anchor": float(np.mean(support["support_distances_to_anchor"])),
    "support_max_distance_to_anchor": float(np.max(support["support_distances_to_anchor"])),
}
support_info

In [ ]:
# Visualize support geometry in the first two PCA directions.

def pca_2d(X):
    X0 = X - X.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(X0, full_matrices=False)
    return X0 @ vh[:2].T

X_vis = np.vstack([support["anchor_vector"][None, :], support["X_support"]])
coords = pca_2d(X_vis)
plt.scatter(coords[1:, 0], coords[1:, 1], label="support candidates")
plt.scatter(coords[0:1, 0], coords[0:1, 1], marker="*", s=180, label="Ball center / anchor")
for i, (x, y) in enumerate(coords[1:]):
    plt.text(x, y, str(i), fontsize=9)
plt.title("One finite support projected to 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

## Run the finite-prior attack on every candidate in the support

For each \(z_i\in S\), we form
\[
D_i=D^-\cup\{z_i\},
\]
release a noisy model from \(D_i\), and let the adversary run the exact finite-prior MAP rule over the same support \(S\).

This gives \(m\) attack trials for Ball-DP and \(m\) attack trials for the standard comparator.

In [ ]:
X_minus, y_minus = remove_train_index(X_train, y_train, support["anchor_index"])

def fit_release_for_attack_dataset(X_full, y_full, *, radius, name):
    release = fit_convex(
        X_full,
        y_full,
        X_eval=X_test,
        y_eval=y_test,
        model_family="ridge_prototype",
        privacy="ball_dp",
        radius=float(radius),
        standard_radius=float(r_std),
        ridge_sensitivity_mode=RIDGE_SENSITIVITY_MODE,
        lam=LAMBDA,
        epsilon=EPSILON,
        delta=DELTA,
        embedding_bound=EMBEDDING_BOUND,
        num_classes=num_classes,
        max_iter=100,
        solver="lbfgs_fullbatch",
        seed=RELEASE_SEED,
    )
    release.extra["demo_name"] = name
    return release

attack_rows = []

for target_pos in range(M):
    x_target = support["X_support"][target_pos]
    y_target = int(support["y_support"][target_pos])

    X_full = np.concatenate([X_minus, x_target[None, :]], axis=0).astype(np.float32)
    y_full = np.concatenate([y_minus, np.asarray([y_target], dtype=np.int32)], axis=0)
    target_index = len(X_full) - 1

    for mechanism, radius in [("Ball-DP", r_ball), ("Standard DP", r_std)]:
        release = fit_release_for_attack_dataset(
            X_full,
            y_full,
            radius=radius,
            name=f"{mechanism} attack target {target_pos}",
        )

        attack, _, _ = attack_convex_ball_output_finite_prior(
            release,
            X_full,
            y_full,
            target_index=target_index,
            X_candidates=support["X_support"],
            y_candidates=support["y_support"],
            prior_weights=None,
            known_label=int(support["anchor_label"]),
            eta_grid=(0.5,),
        )

        diagnostics = diagnose_convex_ball_output_finite_prior(
            release,
            X_full,
            y_full,
            target_index=target_index,
            X_candidates=support["X_support"],
            y_candidates=support["y_support"],
            prior_weights=None,
            known_label=int(support["anchor_label"]),
            center_features=support["anchor_vector"],
            center_label=int(support["anchor_label"]),
        )

        attack_rows.append({
            "mechanism": mechanism,
            "target_position": target_pos,
            "predicted_position": int(attack.diagnostics["predicted_prior_index"]),
            "exact_id_success": float(attack.metrics["exact_identification_success"]),
            "prior_rank": float(attack.metrics["prior_rank"]),
            "posterior_top1_probability": float(attack.metrics["posterior_top1_probability"]),
            "posterior_true_probability": float(attack.metrics["posterior_true_probability"]),
            "posterior_effective_candidates": float(attack.metrics["posterior_effective_candidates"]),
            "log_score_gap_top2": float(attack.metrics["log_score_gap_top2"]),
            "instance_bound_finite_opt": float(diagnostics["bound_direct_instance_finite_opt"]),
            "instance_bound_center_max": float(diagnostics["bound_direct_instance_center_max"]),
            "model_pairwise_snr_median": float(diagnostics["model_pairwise_snr_median"]),
            "model_nn_snr_median": float(diagnostics["model_nn_snr_median"]),
            "ridge_tau": float(diagnostics["ridge_inverse_noise_tau"]),
            "ridge_count_dilution": float(diagnostics["ridge_count_dilution"]),
            "ridge_feature_pairwise_snr_median": float(diagnostics["ridge_feature_pairwise_snr_median"]),
            "sigma": float(release.privacy.ball.sigma),
            "accuracy": float(release.utility_metrics.get("accuracy", np.nan)),
        })

attack_df = pd.DataFrame(attack_rows)
display(attack_df)

In [ ]:
summary = (
    attack_df
    .groupby("mechanism", as_index=False)
    .agg(
        empirical_exact_id=("exact_id_success", "mean"),
        mean_rank=("prior_rank", "mean"),
        posterior_eff_candidates=("posterior_effective_candidates", "mean"),
        posterior_top1_probability=("posterior_top1_probability", "mean"),
        instance_bound_finite_opt=("instance_bound_finite_opt", "mean"),
        instance_bound_center_max=("instance_bound_center_max", "mean"),
        model_pairwise_snr_median=("model_pairwise_snr_median", "mean"),
        model_nn_snr_median=("model_nn_snr_median", "mean"),
        ridge_tau=("ridge_tau", "mean"),
        ridge_count_dilution=("ridge_count_dilution", "mean"),
        sigma=("sigma", "mean"),
        accuracy=("accuracy", "mean"),
    )
)
summary["oblivious_baseline"] = 1.0 / M
display(summary)

In [ ]:
x = np.arange(len(summary))
width = 0.25

plt.bar(x - width, summary["empirical_exact_id"], width=width, label="empirical exact-ID")
plt.bar(x, summary["instance_bound_finite_opt"], width=width, label="instance finite bound")
plt.bar(x + width, summary["oblivious_baseline"], width=width, label="baseline 1/m")
plt.xticks(x, summary["mechanism"])
plt.ylim(0, 1)
plt.ylabel("Probability")
plt.title("Exact-ID attack: empirical success vs theorem-backed bounds")
plt.legend()
plt.show()

## How to read the attack diagnostics

- `empirical_exact_id`: fraction of targets in \(S\) correctly identified.
- `oblivious_baseline`: \(1/m\), the best attack that ignores the release.
- `instance_bound_finite_opt`: finite-support Gaussian upper bound using the actual candidate model means.
- `model_pairwise_snr_median`: median pairwise distance between candidate model means, divided by \(\sigma\).
- `model_nn_snr_median`: median nearest-neighbor model-space separation, divided by \(\sigma\).
- `ridge_tau`: noise scale after the known-label ridge inverse \(W_y=Z+\eta_y\).
- `ridge_count_dilution`: ratio between the target-class count-aware ridge sensitivity and the count-worst-case ridge sensitivity.

A near-flat posterior has high `posterior_effective_candidates`; a confident posterior has lower effective candidate count.

In [ ]:
# Optional: inspect one mechanism's per-target predictions.
display(
    attack_df[[
        "mechanism",
        "target_position",
        "predicted_position",
        "exact_id_success",
        "prior_rank",
        "posterior_true_probability",
        "posterior_top1_probability",
    ]].sort_values(["mechanism", "target_position"])
)

## Relation to the official scripts

This notebook is a single-run demonstration.

The official experiment scripts add:

- multiple datasets,
- multiple radii,
- multiple \(\varepsilon\) values,
- multiple candidate-set sizes \(m\),
- multiple release seeds,
- multiple support anchors and support draws,
- caching and aggregation,
- publication tables and figures.

The mathematical attack is the same finite-prior MAP rule used here.